### Dataset 

| Case | Generative model | Mean | Covariance |
|-----:|------------------|------|------------|
| **Uncorrelated (i.i.d.)** | $ y_i \sim \mathcal N(0,\sigma^2),\; i=1,\dots,n $ | $ \mathbb E[y]=0 $ | $ \mathrm{Cov}(y)=\sigma^2 I $ |
| **Correlated (GP)** | $ y = L\varepsilon,\; \varepsilon\sim\mathcal N(0,I) $ | $ \mathbb E[y]=0 $ | $ \mathrm{Cov}(y)=K+\sigma_n^2 I \equiv K_y $ |

In [1]:
import numpy as np
import ROOT
from IPython.display import display, Image

def generate_and_plot_datasets(
    *,
    n=40,
    x_min=0.0,
    x_max=10.0,
    seed=7,
    sigma_iid=1.0,          # iid noise std
    kernel_fn=None,         # kernel_fn(x, x) -> (n,n) covariance
    sigma_corr_noise=0.2,   # measurement noise std (diagonal)
    jitter=1e-10,           # tiny diagonal stabilizer
    canvas_name="c",
    canvas_title="Correlated vs Non-correlated",
    out_png="gp_toy.png",   # file to save
    show_in_notebook=True,  # display PNG in Jupyter
):
    if kernel_fn is None:
        raise ValueError("kernel_fn must be provided (callable returning an (n,n) covariance matrix).")

    rng = np.random.default_rng(seed)

    # Inputs
    x = np.linspace(x_min, x_max, n)

    # Dataset A: i.i.d.
    y_uncorr = rng.normal(0.0, sigma_iid, size=n)

    # Dataset B: correlated via explicit Cholesky
    K = np.asarray(kernel_fn(x, x), dtype=np.float64)
    if K.shape != (n, n):
        raise ValueError(f"kernel_fn returned shape {K.shape}, expected {(n, n)}")

    # Defensive symmetrize
    K = 0.5 * (K + K.T)

    # Observed covariance
    K_y = K + (sigma_corr_noise**2) * np.eye(n, dtype=np.float64)
    K_y += jitter * np.eye(n, dtype=np.float64)

    L = np.linalg.cholesky(K_y)
    eps = rng.normal(0.0, 1.0, size=n)
    y_corr = L @ eps

    # ROOT plotting
    ROOT.gStyle.SetOptStat(0)
    ROOT.gROOT.SetBatch(True)  # important for notebooks (prevents GUI issues)

    c = ROOT.TCanvas(canvas_name, canvas_title, 1100, 450)
    c.Divide(2, 1)

    def make_graph(xarr, yarr, name):
        g = ROOT.TGraph(len(xarr), xarr.astype(np.float64), yarr.astype(np.float64))
        g.SetName(name)
        g.SetMarkerStyle(20)
        g.SetMarkerSize(1.0)
        return g

    # Left
    c.cd(1)
    g1 = make_graph(x, y_uncorr, "g_uncorr")
    g1.SetTitle(f"Dataset A: i.i.d. noise (sigma={sigma_iid});x;y")
    g1.Draw("AP")

    # Right
    c.cd(2)
    g2 = make_graph(x, y_corr, "g_corr")
    g2.SetTitle(f"Dataset B: correlated (sigma_n={sigma_corr_noise});x;y")
    g2.Draw("AP")

    c.Update()

    # Save + show in notebook
    c.SaveAs(out_png)

    display(Image(filename=out_png))

    return x, y_uncorr, y_corr, c

/opt/homebrew/Cellar/root/6.38.00/lib/root/cppyy/__init__.py:374: UserWarning: CPyCppyy API not found (tried: /Users/chanayo/.pyenv/versions/3.14.0/include/site/python3.14); set CPPYY_API_PATH envar to the 'CPyCppyy' API directory to fix
  warnings.warn("CPyCppyy API not found (tried: %s); "


## Kernel 

$$
k(x_i, x_j)
=
\sigma_f^2
\exp\!\left(
-\frac{(x_i - x_j)^2}{2\ell^2}
\right),
$$

$$
K_{ij} = k(x_i, x_j).
$$

In [2]:
# kernel (SE/RBF)
def rbf_kernel(x1, x2, sigma_f=1.5, ell=1.2):
    x1 = np.asarray(x1)[:, None]
    x2 = np.asarray(x2)[None, :]
    sqdist = (x1 - x2) ** 2
    return (sigma_f**2) * np.exp(-0.5 * sqdist / (ell**2))


In [23]:
import numpy as np

def generate_datasets(
    *,
    n=40,
    x_min=0.0,
    x_max=10.0,
    seed=7,
    sigma_iid=1.0,          # iid noise std
    kernel_fn=None,         # kernel_fn(x, x) -> (n,n) covariance
    sigma_corr_noise=0.2,   # measurement noise std (diagonal)
    jitter=1e-10,           # tiny diagonal stabilizer
):
    if kernel_fn is None:
        raise ValueError("kernel_fn must be provided (callable returning an (n,n) covariance matrix).")

    rng = np.random.default_rng(seed)

    # Inputs
    x = np.linspace(x_min, x_max, n)

    # Dataset A: i.i.d.
    y_uncorr = rng.normal(0.0, sigma_iid, size=n)

    # Dataset B: correlated via Cholesky
    K = np.asarray(kernel_fn(x, x), dtype=np.float64)
    if K.shape != (n, n):
        raise ValueError(f"kernel_fn returned shape {K.shape}, expected {(n, n)}")

    # Defensive symmetrization
    K = 0.5 * (K + K.T)

    # Observed covariance
    K_y = K + (sigma_corr_noise**2) * np.eye(n)
    K_y += jitter * np.eye(n)

    # Sample
    L = np.linalg.cholesky(K_y)
    eps = rng.normal(0.0, 1.0, size=n)
    y_corr = L @ eps

    return x, y_uncorr, y_corr, K_y

## Kernels

In [24]:
# -------------------------
# Squared Exponential / RBF
# -------------------------
def rbf_kernel(x1, x2, sigma_f=1.5, ell=1.2):
    x1 = np.asarray(x1)[:, None]
    x2 = np.asarray(x2)[None, :]
    return (sigma_f**2) * np.exp(-0.5 * (x1 - x2)**2 / ell**2)


# -------------------------
# Linear kernel
# k(x, x') = sigma_f^2 (x - c)(x' - c)
# -------------------------
def linear_kernel(x1, x2, sigma_f=1.0, c=0.0):
    x1 = np.asarray(x1)[:, None]
    x2 = np.asarray(x2)[None, :]
    return sigma_f**2 * (x1 - c) * (x2 - c)


# -------------------------
# Polynomial kernel
# -------------------------
def polynomial_kernel(x1, x2, sigma_f=1.0, degree=2, c=1.0):
    x1 = np.asarray(x1)[:, None]
    x2 = np.asarray(x2)[None, :]
    return sigma_f**2 * (x1 @ x2.T + c) ** degree


# -------------------------
# White noise kernel
# k(x, x') = sigma_n^2 δ(x - x')
# -------------------------
def white_kernel(x1, x2, sigma_n=1.0):
    x1 = np.asarray(x1)
    x2 = np.asarray(x2)
    if len(x1) != len(x2):
        raise ValueError("White kernel requires x1 and x2 of same length")
    return sigma_n**2 * np.eye(len(x1))


In [31]:
import ROOT
import numpy as np
import uuid

def plot_datasets_root_inline(
    x,
    y_uncorr,
    y_corr,
    *,
    title_left="Dataset A: i.i.d.",
    title_right="Dataset B: correlated",
    canvas_name="c",
    canvas_title="Correlated vs Non-correlated",
    width=1100,
    height=450,
):
    # ROOT config: inline only (no GUI window)
    ROOT.gROOT.SetBatch(True)
    ROOT.gStyle.SetOptStat(0)

    # Avoid canvas name collisions
    uid = uuid.uuid4().hex[:6]
    cname = f"{canvas_name}_{uid}"

    c = ROOT.TCanvas(cname, canvas_title, width, height)
    c.Divide(2, 1)

    # Keep references alive (CRITICAL for JSROOT)
    c._graphs = []

    def make_graph(xarr, yarr, name):
        g = ROOT.TGraph(
            len(xarr),
            np.asarray(xarr, dtype=np.float64),
            np.asarray(yarr, dtype=np.float64),
        )
        g.SetName(f"{name}_{uid}")
        g.SetMarkerStyle(20)
        g.SetMarkerSize(1.0)
        return g

    # -------- Left: i.i.d. --------
    c.cd(1)
    g1 = make_graph(x, y_uncorr, "g_uncorr")
    g1.SetTitle(f"{title_left};x;y")
    g1.Draw("AP")
    c._graphs.append(g1)

    # -------- Right: correlated --------
    c.cd(2)
    g2 = make_graph(x, y_corr, "g_corr")
    g2.SetTitle(f"{title_right};x;y")
    g2.Draw("AP")
    c._graphs.append(g2)

    c.Modified()
    c.Update()

    # JSROOT inline render (THIS is the key)
    if hasattr(ROOT, "JSROOT") and hasattr(ROOT.JSROOT, "Draw"):
        return ROOT.JSROOT.Draw(c)

    # Fallback (some environments auto-render)
    return c

In [32]:
# Parametros

sigma_iid = 0.5
sigma_corr = 0.0

def kernel_fn(a, b):
    return rbf_kernel(a, b, sigma_f=1.5, ell=1.2)

# generate
x, y_uncorr, y_corr, K_y = generate_datasets(
    sigma_iid=sigma_iid,
    kernel_fn=kernel_fn,
    sigma_corr_noise=sigma_corr,
)

# plot
c = plot_datasets_root_inline(x, y_uncorr, y_corr)
